#  Visualizing London Air Quality Data

---  
## 1. Objectives

This notebook analyzes and visualizes air quality data in London.  

> 
> **Data sources** 
>
> The data for this analysis is courtesy of the Environmental Research Group at Imperial College London.  
>
> You can read more about their work here: https://www.imperial.ac.uk/school-public-health/environmental-research-group/ 
>
> We used their API to obtain information about monitoring sites and a subset of recent monitoring data.  
>
> You can read more about the API here: https://api.erg.ic.ac.uk/AirQuality/help
>

Automatic monitoring stations located around London continuously measure a range of pollutants including:

- Carbon Monoxide (CO)
- Nitrogen Dioxide (NO₂) 
- Ozone (O₃)
- Particulate Matter (PM10 and PM2.5)
- Sulphur Dioxide (SO₂)

Here's a typical monitoring site, code CE3 on Regent Street in London's West End.

![Monitoring site CE3](./IMG_1952.jpeg)

Not all sites monitor all pollutants, and sites tend not to have indefinite operation lifetimes.

The analysis explores:  

- Monitoring sites: their locations, types, and operational timelines.  

- Species information: details about the different pollutants monitored and their health effects.  

- Measurement data: pollutant measurements across London, their temporal variations and correlations.  


---  
## 2. Monitoring sites

#### 2.1 Getting started

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'vscode'
pio.templates.default = 'plotly'


#### 2.2 Read sites data

In [2]:
df_sites = pd.read_parquet('./data/laq-sites.parquet')
df_sites.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   LocalAuthorityCode  258 non-null    int64         
 1   LocalAuthorityName  258 non-null    object        
 2   SiteCode            258 non-null    object        
 3   SiteName            258 non-null    object        
 4   SiteType            258 non-null    object        
 5   DateClosed          174 non-null    datetime64[ms]
 6   DateOpened          258 non-null    datetime64[ms]
 7   Latitude            256 non-null    float64       
 8   Longitude           256 non-null    float64       
 9   DataOwner           258 non-null    object        
 10  DataManager         258 non-null    object        
 11  SiteLink            258 non-null    object        
 12  SiteActive          258 non-null    bool          
dtypes: bool(1), datetime64[ms](2), float64(2), int64(1

#### 2.3 Location map of all sites

In [3]:
fig23 = px.scatter_map(
    data_frame=df_sites,
    lon="Longitude",
    lat="Latitude",
    text="SiteCode",
    color="SiteType",
)

fig23.update_layout(
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    legend=dict(
        x=0.98,  
        y=0.02,
        xanchor="right", 
        bgcolor="rgba(255,255,255,0.8)", 
        bordercolor="rgba(0,0,0,0)", 
        borderwidth=1,
    ),
    annotations=[
        dict(
            text="London Air Quality Monitoring Sites",
            x=0.02, 
            y=0.98,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=24, color="black"),
        )
    ],
)

fig23.add_scattermap(
    legendgroup="Locations",
    legendgrouptitle=dict(text="<br>Locations"),
    name="Meetup",
    lon=[-0.08969],
    lat=[51.52147],
    mode="markers+text",
    marker=dict(color="midnightblue"),
    text=["Meetup"]
)

fig23.update_traces(
    textposition="top left", textfont_color="black", textfont_size=16, marker_size=15
)

fig23.show()

#### 2.4 Timeline of active sites

In [4]:
from datetime import datetime

df_sites_timeline = df_sites[df_sites['SiteActive'] == True]

x_start = df_sites_timeline['DateOpened']
x_end   = df_sites_timeline["DateClosed"].fillna(datetime.now())

df_sites.sort_values(
    by=["LocalAuthorityName", "DateOpened"], ascending=[False, False], inplace=True
)

fig24 = px.timeline(
    data_frame=df_sites_timeline,
    x_start="DateOpened",
    x_end=x_end,
    y="SiteCode",
    color="LocalAuthorityName",
    title="Active Sites Timeline",
    height=800
)

fig24.update_layout(
    xaxis_rangeslider_visible=True,
    yaxis_fixedrange=False
)

fig24.show()

---  
## 3. Measurement data

#### 3.1 What species are measured?

In [5]:
df_species = pd.read_csv('./data/laq-species.csv')
df_species

,SpeciesCode,SpeciesName,Description,HealthEffect,Link
0,CO,Carbon Monoxide,"Carbon Monoxide is a colourless, odourless poi...",The gas affects the transport of oxygen around...,http://www.londonair.org.uk/LondonAir/guide/Wh...
1,NO2,Nitrogen Dioxide,Nitrogen oxides are formed during high tempera...,Nitrogen Dioxide has several health impacts an...,http://www.londonair.org.uk/LondonAir/guide/Wh...
2,O3,Ozone,"Ozone is not directly emitted, but is formed b...","Like nitrogen dioxide, high levels of ozone ca...",http://www.londonair.org.uk/LondonAir/guide/Wh...
3,PM10,PM10 Particulate,Larger particles in the atmosphere are general...,The effects of inhaling particulate matter hav...,http://www.londonair.org.uk/LondonAir/guide/Wh...
4,PM25,PM2.5 Particulate,These are particles which are less than 2.5 mi...,They are thought to have a greater effect on h...,http://www.londonair.org.uk/LondonAir/guide/Wh...
5,SO2,Sulphur Dioxide,Sulphur Dioxide is produced when a material or...,Short-term exposure to high levels of sulphur ...,http://www.londonair.org.uk/LondonAir/guide/Wh...


#### 3.2 Read measurement data

In [6]:
df_data = pd.read_parquet('./data/laq-data-30.parquet')
df_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46897 entries, 0 to 46896
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   SiteCode            46897 non-null  object        
 1   MeasurementDateGMT  46897 non-null  datetime64[ns]
 2   CO                  1223 non-null   float64       
 3   NO2                 42501 non-null  float64       
 4   O3                  7656 non-null   float64       
 5   PM10                29546 non-null  float64       
 6   PM25                26560 non-null  float64       
 7   SO2                 2354 non-null   float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 2.9+ MB


#### 3.3 Summary statistics of ozone (O3) meaurements

In [7]:
# 
# find rows where O3 is measured:
#
df_o3 = df_data.dropna(subset=['O3']).copy()

#
# find unique sites that measure O3:
#
o3_sites = df_o3['SiteCode'].unique()
print(f"Number of sites measuring O3: {len(o3_sites)}")
print("Sites with O3 measurements:", o3_sites)

#
# summary statistics of O3 measurements by site:
#
o3_by_site = df_o3.groupby('SiteCode').O3.describe().round(2)

#
# sort by median (50%):
#
o3_by_site.sort_values(by='50%', ascending=False, inplace=True)

print("\nO3 summary statistics by site, sorted by median:")
o3_by_site


Number of sites measuring O3: 13
Sites with O3 measurements: ['BL0' 'BQ7' 'BX1' 'CE2' 'GB6' 'GN3' 'HG4' 'HP1' 'KC1' 'MY1' 'RI2' 'TH6'
 'WME']

O3 summary statistics by site, sorted by median:


,count,mean,std,min,25%,50%,75%,max
SiteCode,,,,,,,,
KC1,714.0,58.84,22.04,7.0,44.95,56.00,70.40,155.5
HP1,712.0,58.54,21.61,16.7,45.78,54.05,69.85,153.5
WME,721.0,57.16,23.92,2.2,42.10,53.40,70.30,160.3
RI2,719.0,55.43,22.53,5.1,41.00,52.70,66.00,156.6
TH6,468.0,54.53,20.88,7.4,40.38,51.65,65.05,139.8
HG4,38.0,53.17,12.20,23.0,44.75,50.30,62.20,81.2
BL0,690.0,52.23,19.78,9.0,40.30,49.80,60.70,140.5
BQ7,721.0,51.04,23.35,-0.2,35.50,47.80,63.40,147.1
BX1,713.0,46.68,21.16,1.0,32.70,43.60,58.70,129.3


#### 3.4 Ozone time series

In [8]:
fig34 = px.line(data_frame=df_o3, x='MeasurementDateGMT', y='O3', color='SiteCode')

fig34.update_layout(
    title="Ozone (O3) Measurements by Site",
    xaxis_title="Date (GMT)",
    yaxis_title="O3 Concentration",
    legend_title="Site Code",
    height=600
)

fig34.show()

#### 3.5 Ozone variation by time of day across all sites

In [9]:
df_o3['Hour'] = df_o3["MeasurementDateGMT"].dt.hour

fig35 = px.box(
    df_o3,
    x="Hour",
    y="O3",
    title="Ozone variation by time of day (all sites)",
    labels={"Hour": "Hour of Day", "O3": "O3 Concentration"},
)
fig35.show()

#### 3.6 Ozone variation by time of day and monitoring site

In [10]:
#
# compute median O3 by hour and site:
#
hourly_median = df_o3.groupby(['Hour', 'SiteCode'])['O3'].median().reset_index()

#
# pivot the data to create a matrix:
#
pivot_data = hourly_median.pivot(index='SiteCode', columns='Hour', values='O3')

#
# create the heatmap:
#
fig36 = px.imshow(
    pivot_data,
    labels=dict(y="Site Code", x="Hour of Day", color="Median O3 Concentration"),
    title="Median O3 Levels by Hour and Site",
    color_continuous_scale="viridis"
)

fig36.show()

#### 3.7 Correlation between species across all sites

In [11]:
# 
# calculate correlation matrix:
#
corr_matrix = df_data.corr(numeric_only=True,)

# 
# create a heatmap visualization:
#
fig37 = px.imshow(
    corr_matrix,
    text_auto='.2f',  
    color_continuous_scale='rdbu',  
    zmin=-1,  
    zmax=1,  
    title="Correlation matrix (all sites)"
)

fig37.update_layout(
    height=600,
    width=600,
)

fig37.show()

---  
laq-visualization.ipynb